# Per-well averages of CellProfiler measurements

CellProfiler writes one CSV file per object type, with one row per object
(cell, nucleus, ...) and one column per measurement. Here we use pandas to

1. load the CSV files,
2. average one measurement per well, and
3. show the averages as a heatmap in the plate layout.

First for an intensity measurement in the cytoplasm and in the nuclei, then
for a shape measurement of the cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Settings

The measurements to look at: change these and run the notebook again
(Run → Run All Cells).

In [ ]:
DATA_DIR = Path("..") / "CellProfiler"   # folder with the CellProfiler CSV files

INTENSITY = "Intensity_MeanIntensity_GFP"   # from MyExpt_Cytoplasm.csv and MyExpt_Nuclei.csv
SHAPE = "AreaShape_FormFactor"              # from MyExpt_Cells.csv

## Load the data

`pd.read_csv` reads a CSV file into a `DataFrame`, a table with named columns.

In [ ]:
cytoplasm = pd.read_csv(DATA_DIR / "MyExpt_Cytoplasm.csv")
nuclei = pd.read_csv(DATA_DIR / "MyExpt_Nuclei.csv")
cells = pd.read_csv(DATA_DIR / "MyExpt_Cells.csv")

print("cytoplasm:", cytoplasm.shape)   # (rows, columns)
print("nuclei:   ", nuclei.shape)
print("cells:    ", cells.shape)

Each row is one object. The `Metadata_` columns tell which image the object
comes from (plate, well, site), the other columns are measurements:

In [ ]:
cytoplasm.head()

All intensity measurements of the cytoplasm (other good starting points:
`"AreaShape"`, `"Texture"`, `"Location"`):

In [ ]:
[c for c in cytoplasm.columns if c.startswith("Intensity")]

## Cytoplasm intensity, step by step

### Average per well

`groupby` splits the rows into groups by the value of a column, here the well.
Then we pick the measurement column and take the mean of each group:

In [ ]:
cytoplasm.groupby("Metadata_Well")[INTENSITY].mean()

`agg` computes several statistics at once. The number of cells per well
(`count`) is worth checking: an average of a few cells is not reliable.

In [ ]:
cytoplasm.groupby("Metadata_Well")[INTENSITY].agg(["mean", "std", "count"])

### Plate layout

For the heatmap we need a table with plate rows (A, B, ...) as rows and plate
columns (1, 2, ...) as columns. We split the well name, e.g. `"B03"`, into its
row letter `"B"` and column number `3`, and then `pivot` the table:

In [ ]:
well_means = cytoplasm.groupby("Metadata_Well")[INTENSITY].mean().reset_index()
well_means["Row"] = well_means["Metadata_Well"].str[0]               # "B03" -> "B"
well_means["Column"] = well_means["Metadata_Well"].str[1:].astype(int)   # "B03" -> 3
well_means.head()

In [ ]:
plate = well_means.pivot(index="Row", columns="Column", values=INTENSITY)
plate

### Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(plate, annot=True, fmt=".4f", cmap="viridis", square=True, ax=ax)
ax.set_title(f"Cytoplasm: {INTENSITY}")
plt.show()

## The same as functions

To repeat this for other objects and measurements, we put the steps into two
functions.

In [ ]:
def plate_layout(df, measurement, statistic="mean"):
    """Per-well statistic ("mean", "median", "count", ...) of a measurement, as a table in the plate
    layout (rows A, B, ... x columns 1, 2, ...)."""
    well_means = df.groupby("Metadata_Well")[measurement].agg(statistic).reset_index()
    well_means["Row"] = well_means["Metadata_Well"].str[0]
    well_means["Column"] = well_means["Metadata_Well"].str[1:].astype(int)
    return well_means.pivot(index="Row", columns="Column", values=measurement)


def plate_heatmap(plate, title, fmt=".4f"):
    """Heatmap of a table from plate_layout(), with the value of each well."""
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(plate, annot=True, fmt=fmt, cmap="viridis", square=True, ax=ax)
    ax.set_title(title)
    plt.show()

## Nuclei intensity

In [ ]:
nuclei.groupby("Metadata_Well")[INTENSITY].agg(["mean", "std", "count"])

In [ ]:
plate = plate_layout(nuclei, INTENSITY)
plate_heatmap(plate, f"Nuclei: {INTENSITY}")

## Cell shape

`AreaShape_FormFactor` is 4π·area/perimeter²: 1 for a circle, smaller for
elongated or irregular cells.

In [ ]:
[c for c in cells.columns if c.startswith("AreaShape")]

In [ ]:
cells.groupby("Metadata_Well")[SHAPE].agg(["mean", "std", "count"])

In [ ]:
plate = plate_layout(cells, SHAPE)
plate_heatmap(plate, f"Cells: {SHAPE}", fmt=".3f")

## Cell counts

The number of cells per well: `"count"` counts the rows of each well that have
a value, so any column without missing values will do, e.g. `ObjectNumber`.
Wells with few cells give less reliable averages above.

In [ ]:
plate = plate_layout(cells, "ObjectNumber", statistic="count")
plate_heatmap(plate, "Number of cells", fmt=".0f")